# Virtual Crystallography Lab
Welcome to the Virtual Crystallography Lab! In this interactive tutorial, we will bridge **synth-pdb** and **synth-xtal** to demonstrate the complete workflow of an X-ray crystallography experiment—entirely in silico.

### Workflow:
1. **Structure Generation:** Generate a perfect $\alpha$-helix using `synth-pdb`.
2. **Diffraction Simulation:** Collect "virtual data" by simulating an MTZ file using `synth-xtal`, complete with bulk solvent modeling.
3. **Phase Errors:** Introduce artificial phase errors to simulate imperfect experimental phasing.
4. **Map Calculation:** Calculate an $F_{obs}$ map and visualize the distorted electron density.

In [ ]:
# Install required packages
!pip install synth-pdb git+https://github.com/elkins-lab/synth-xtal.git reciprocalspaceship gemmi py3Dmol

### 1. Generating the Virtual Crystal
First, let's create a 30-residue alpha-helix using the `synth-pdb` suite.

In [ ]:
!synth-pdb --length 30 --conformation alpha --output virtual_protein.pdb

### 2. Simulating the Diffraction Experiment
Now we shoot our virtual protein with X-rays! We use `synth-xtal` to calculate the structure factors (amplitudes and phases). We enable `use_bulk_solvent` to simulate the disordered water surrounding the protein, which is critical for realistic low-resolution intensities.

In [ ]:
from synth_xtal.simulator import simulate_diffraction

simulate_diffraction(
    input_pdb="virtual_protein.pdb",
    output_mtz="virtual_data.mtz",
    d_min=2.0,
    margin=15.0,
    use_bulk_solvent=True,
)

### 3. Introducing Experimental Phase Errors
In a real experiment, you can only measure the intensities ($|F|^2$), and you lose the phase information. Phasing algorithms (like Molecular Replacement or SAD) attempt to recover the phases, but they are never perfect.

Let's simulate this by taking our perfect calculated phases (`PHIC`) and adding random noise based on resolution.

In [ ]:
import reciprocalspaceship as rs
import numpy as np

# Load the simulated data
ds = rs.read_mtz("virtual_data.mtz")

# Add random phase errors (larger errors at higher resolution)
ds.compute_dHKL(inplace=True)
error_magnitude = 60.0 * (1.0 / ds["dHKL"])  # Up to 30 degrees at 2.0A
phase_noise = np.random.normal(0, error_magnitude.to_numpy())

ds["PHI_OBS"] = ds["PHIC"] + phase_noise
ds.infer_mtz_dtypes(inplace=True)

ds.write_mtz("experimental_data.mtz")
print("Saved experimental_data.mtz with phase noise.")

### 4. Visualizing the Distorted Map
Finally, let's use `gemmi` to FFT our "experimental" amplitudes and noisy phases back into real space to see what a crystallographer would look at in Coot!

In [ ]:
import gemmi
import matplotlib.pyplot as plt

mtz = gemmi.read_mtz_file("experimental_data.mtz")
grid = mtz.transform_f_phi_to_map("FC", "PHI_OBS", sample_rate=3.0)

# Plot a 2D slice of the electron density
arr = np.array(grid, copy=False)
z_slice = arr.shape[2] // 2

plt.figure(figsize=(8, 8))
plt.imshow(arr[:, :, z_slice], cmap="viridis", origin="lower")
plt.title("Simulated Electron Density Map Slice")
plt.colorbar(label="e- / A^3")
plt.show()